# Systemy AI – Laboratorium 3
## Titanic: wybieranie danych, braki, tabele przestawne i pierwszy klasyfikator

**Cel:** przygotować prawdziwy zbiór danych do modelowania (wybieranie wierszy i kolumn, braki danych, grupowanie), zwizualizować go i zbudować pierwszy klasyfikator, który „uczy się” z danych.


---
## Przygotowanie

Komórka poniżej wczytuje biblioteki i dane oraz definiuje funkcję `binary_metrics` z Lab 1 (tym razem gotową). Uruchom ją i nic w niej nie zmieniaj.

Zbiór opisuje 891 pasażerów Titanica. Zadanie klasyfikacji binarnej: przewidzieć kolumnę `survived`.

| Kolumna | Znaczenie |
|---|---|
| `survived` | **etykieta**: 1 = przeżył/a, 0 = nie |
| `pclass` | klasa podróży: 1, 2 lub 3 |
| `sex` | płeć: `"male"` / `"female"` |
| `age` | wiek w latach (są braki!) |
| `fare` | cena biletu |
| `embarked` | port zaokrętowania: S = Southampton, C = Cherbourg, Q = Queenstown (są braki!) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = sns.load_dataset("titanic")
df = df[["survived", "pclass", "sex", "age", "fare", "embarked"]].copy()


def binary_metrics(y_true, y_pred):
    """Metryki klasyfikacji binarnej (z Lab 1). Metryka z zerowym mianownikiem = 0.0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    div = lambda a, b: a / b if b > 0 else 0.0
    precision, recall = div(TP, TP + FP), div(TP, TP + FN)
    return {"TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "accuracy": div(TP + TN, len(y_true)), "precision": precision,
            "recall": recall, "f1": div(2 * precision * recall, precision + recall)}


print(df.shape)
df.head()

---
## Część A. pandas: `loc` i `iloc`

Wiersze i kolumny `DataFrame` można wybierać na dwa sposoby:

| | `df.iloc[...]` | `df.loc[...]` |
|---|---|---|
| wybiera po | **pozycji** (0, 1, 2, …), jak w liście | **etykiecie** indeksu i **nazwie** kolumny |
| przykład | `df.iloc[0, 3]` – 1. wiersz, 4. kolumna | `df.loc[0, "age"]` – wiersz o indeksie 0, kolumna `age` |
| wycinek `5:7` | wiersze na pozycjach 5 i 6 (**bez** 7) | wiersze o etykietach 5, 6 **i 7** |
| warunek logiczny | nie | tak: `df.loc[df["age"] < 10, "survived"]` |

Warunki łączymy operatorami `&` (i), `|` (lub), `~` (nie), a **każdy warunek bierzemy w nawias**:
`df.loc[(df["sex"] == "female") & (df["pclass"] == 3)]`.

Zaraz po wczytaniu indeks to 0, 1, 2, …, więc pozycja i etykieta są takie same. Różnica wychodzi po filtrowaniu albo sortowaniu: wtedy indeks ma „dziury” i `iloc[0]` nie jest już wierszem o etykiecie 0.

### Zadanie A1. Wybieranie danych
1. `first_row` – pierwszy wiersz tabeli (`iloc`).
2. `corner` – pierwsze 5 wierszy i pierwsze 3 kolumny (`iloc`).
3. `ages_5_7` – kolumna `age` dla wierszy o etykietach od 5 do 7 włącznie (`loc`).
4. `women3` – wszystkie kobiety podróżujące 3. klasą (`loc` z warunkiem) oraz `p_women3` – odsetek ocalałych w tej grupie.
5. `old_fare` – ceny biletów (tylko kolumna `fare`) pasażerów w wieku powyżej 70 lat.

W komentarzu: ile wierszy zwraca `df.iloc[5:7]`, a ile `df.loc[5:7]`? Dlaczego?

In [ ]:
# Twoje rozwiązanie
# df.iloc[5:7] => daje 2 wiersze, df.loc[5:7] => daje 3 wiersze. Jest tak bo w iloc prawy koniec jest skracany jak w pythonie, a w przypadku loc bo przy etykietach prawy koniec jest włączony.
first_row = df.iloc[0]
corner = df.iloc[0:5, 0:3]
ages_5_7 = df.loc[5:7, "age"]
women3 = df.loc[(df["sex"] == "female") & (df["pclass"] == 3)]
p_women3 = len(women3.loc[(women3["survived"] == 1)]) / len(women3)
old_fare = df.loc[(df["age"] > 70), "fare"]

In [ ]:
# TEST
assert first_row["age"] == 22.0 and first_row["sex"] == "male"
assert corner.shape == (5, 3) and list(corner.columns) == ["survived", "pclass", "sex"]
assert list(ages_5_7.index) == [5, 6, 7]
assert len(women3) == 144 and abs(p_women3 - 0.5) < 1e-9
assert len(old_fare) == 5 and old_fare.name == "fare"
print("A1 OK")

---
## Część B. Braki danych

Brakująca wartość w pandas to `NaN` (*Not a Number*). Większość modeli uczenia maszynowego **nie przyjmie** tabeli z brakami, więc trzeba je wykryć i coś z nimi zrobić.

| Strategia | Kod | Kiedy | Wada |
|---|---|---|---|
| usuń wiersze | `df.dropna(subset=["age"])` | braków jest bardzo mało | tracimy dane; jeśli braki nie są losowe, zbiór staje się stronniczy |
| średnia | `s.fillna(s.mean())` | rozkład symetryczny, bez wartości odstających | wrażliwa na wartości odstające |
| mediana | `s.fillna(s.median())` | rozkład skośny lub z wartościami odstającymi | wszystkie braki dostają tę samą wartość |
| moda | `s.fillna(s.mode()[0])` | cechy **kategoryczne** (np. port) | wzmacnia najczęstszą kategorię |
| mediana w grupach | `s.fillna(df.groupby(["sex", "pclass"])["age"].transform("median"))` | wartość zależy od innych cech | trochę więcej kodu |
| flaga braku | `df["age_missing"] = df["age"].isna().astype(int)` | sam fakt braku może coś znaczyć | dodatkowa kolumna |

Każde uzupełnienie to **zgadywanie**. Uzupełnione wartości nie są pomiarami, tylko naszym założeniem, i zmniejszają zmienność danych.

**Najważniejsze pytanie:** czy braki są losowe? Jeśli wiek częściej brakuje u pasażerów 3. klasy, to braki niosą informację, a usunięcie tych wierszy zmienia strukturę zbioru.

### Zadanie B1. Gdzie są braki i czy są losowe?
1. `missing` – liczba braków w każdej kolumnie (`isna().sum()`).
2. `frac_age` – jaki **ułamek** wierszy nie ma wieku (`isna().mean()`).
3. `miss_by_class` – ułamek braków wieku w każdej klasie podróży (pogrupuj po `pclass` kolumnę `df["age"].isna()`).
4. `surv_by_missing` – odsetek ocalałych wśród osób **z** podanym wiekiem i **bez** niego (`df.groupby(df["age"].isna())["survived"].mean()`).

In [ ]:
# Twoje rozwiązanie
missing = df.isna().sum()
frac_age = df["age"].isna().mean()
miss_by_class = df["age"].isna().groupby(df["pclass"]).mean()
surv_by_missing = df.groupby(df["age"].isna())["survived"].mean()
print(missing)
print(frac_age)
print(miss_by_class)
print(surv_by_missing)

In [ ]:
# TEST
assert missing["age"] == 177 and missing["embarked"] == 2 and missing["fare"] == 0
assert abs(frac_age - 0.1987) < 1e-3
assert abs(miss_by_class[3] - 0.277) < 1e-3 and abs(miss_by_class[2] - 0.060) < 1e-3
assert abs(surv_by_missing[True] - 0.294) < 1e-3 and abs(surv_by_missing[False] - 0.406) < 1e-3
print("B1 OK")

*Twoja odpowiedź:* czy braki wieku są losowe? Uzasadnij liczbami.
Braki wieku nie są losowe. Odsetek braków różni się silnie między klasami (13,9% w 1. klasie, 6,0% w 2., 27,7% w 3.), choć przy losowych brakach wszędzie byłoby ok. 19,9%. Dodatkowo wśród pasażerów bez wieku przeżyło 29,4%, a wśród tych z wiekiem 40,6%. Braki zależą więc od klasy i wiążą się z przeżyciem, więc proste usunięcie tych wierszy lub wstawienie jednej średniej mogłoby zniekształcić wyniki.

### Zadanie B2. Porównanie strategii
Przygotuj cztery wersje kolumny `age`:
- `age_drop` – braki usunięte (`dropna()`),
- `age_mean` – braki uzupełnione średnią,
- `age_median` – braki uzupełnione medianą,
- `age_group` – braki uzupełnione medianą wieku **w grupie (płeć, klasa)** (`groupby(...)["age"].transform("median")`; `transform` zwraca wartość grupy dla każdego wiersza, więc wynik ma tyle samo wierszy co `df`).

Zbierz w tabeli `cmp` (wiersze: `drop`, `mean`, `median`, `group`) kolumny `n` (liczba wartości), `mean` i `std`.

Następnie porównaj średnią i medianę **ceny biletu** (`fare`): zapisz je w `fare_mean` i `fare_median`.

In [ ]:
# Twoje rozwiązanie
age_drop = df["age"].dropna()
age_mean = df["age"].fillna(df["age"].mean())
age_median = df["age"].fillna(df["age"].median())
age_group = df["age"].fillna(
    df.groupby(["sex", "pclass"])["age"].transform("median")
)

series = {
    "drop": age_drop,
    "mean": age_mean,
    "median": age_median,
    "group": age_group,
}

cmp = pd.DataFrame({
    name: {"n": s.count(), "mean": s.mean(), "std": s.std()}
    for name, s in series.items()
}).T

fare_mean = df["fare"].mean()
fare_median = df["fare"].median()

print(cmp)

In [ ]:
# TEST
assert list(cmp.index) == ["drop", "mean", "median", "group"]
assert list(cmp["n"]) == [714, 891, 891, 891]
assert abs(cmp.loc["median", "mean"] - 29.362) < 1e-2 and abs(cmp.loc["group", "mean"] - 29.112) < 1e-2
assert abs(cmp.loc["drop", "std"] - 14.526) < 1e-2 and abs(cmp.loc["mean", "std"] - 13.002) < 1e-2
assert abs(fare_mean - 32.20) < 1e-2 and fare_median == 14.4542
print("B2 OK")

*Twoja odpowiedź:*
1. Dlaczego po uzupełnieniu (średnią lub medianą) odchylenie standardowe wieku spada?
2. Dla ceny biletu średnia i mediana bardzo się różnią. Dlaczego? Którą wartością uzupełnił(a)byś braki w `fare`?
3. Która strategia dla `age` wydaje Ci się najrozsądniejsza i dlaczego?

1. Po uzupełnieniu średnią lub medianą odchylenie standardowe spada, bo do danych dodajemy wiele identycznych wartości położonych w środku rozkładu (lub tuż obok), czyli o zerowym lub minimalnym odchyleniu od średniej. Rozrzut zostaje sztucznie zaniżony. Uzupełnianie medianą w grupach (płeć, klasa) zaniża go mniej, bo wstawia zróżnicowane wartości.
2. Średnia i mediana bardzo się różnią, bo rozkład ceny jest prawoskośny: nieliczne bardzo drogie bilety zawyżają średnią, a mediana jest na nie odporna. Braki w fare uzupełniłbym medianą (najlepiej w obrębie klasy), bo lepiej odzwierciedla typową cenę.
3. Wybrałbym uzupełnienie medianą w grupach (płeć, klasa). Braki nie są losowe, więc drop zmieniłby skład próby (usunąłby głównie pasażerów 3. klasy). Uzupełnienie grupowe zachowuje wszystkie wiersze i zaniża rozrzut mniej niż jedna stała. Dodatkowo warto zachować kolumnę age_missing, bo sam fakt braku wieku niesie informację o przeżyciu.

### Zadanie B3. Czysty zbiór
Utwórz `df_clean = df.copy()` i w nim:
1. dodaj kolumnę `age_missing` (1, jeśli wiek był brakujący, w przeciwnym razie 0) – **zanim** uzupełnisz braki;
2. uzupełnij `age` medianą w grupach (płeć, klasa);
3. uzupełnij `embarked` **modą**.

Na koniec sprawdź, że w `df_clean` nie ma już żadnych braków. Od tej chwili pracujemy na `df_clean`.

In [ ]:
# Twoje rozwiązanie
df_clean = df.copy()
df_clean["age_missing"] = df_clean["age"].isna().astype(int)
df_clean["age"] = df_clean["age"].fillna(
    df_clean.groupby(["sex", "pclass"])["age"].transform("median")
)
df_clean["embarked"] = df_clean["embarked"].fillna(df_clean["embarked"].mode()[0])

print(df_clean)

In [ ]:
# TEST
assert df_clean.isna().sum().sum() == 0, "W df_clean są jeszcze braki"
assert df_clean["age_missing"].sum() == 177, "Flagę braku trzeba dodać przed uzupełnieniem"
assert df["age"].isna().sum() == 177, "Oryginalny df nie powinien się zmienić - użyj .copy()"
assert df_clean["embarked"].value_counts()["S"] == 646
assert abs(df_clean["age"].mean() - 29.112) < 1e-2
print("B3 OK")

---
## Część C. Grupowanie i tabela przestawna

`groupby` działa w trzech krokach: **podziel** wiersze na grupy → **policz** coś w każdej grupie → **złóż** wyniki w tabelę.

Średnia z kolumny 0/1 to odsetek jedynek, więc `df_clean.groupby("sex")["survived"].mean()` to odsetek ocalałych w każdej płci.

**Tabela przestawna** (`pivot_table`) to grupowanie po **dwóch** cechach naraz, ułożone w siatkę: jedna cecha w wierszach, druga w kolumnach, a w komórkach wynik funkcji `aggfunc` (np. `"mean"`, `"count"`, `"median"`):

```python
df_clean.pivot_table(index="sex", columns="pclass", values="survived", aggfunc="mean")
```

### Zadanie C1. Kto przeżył?
1. `surv_by_sex` – odsetek ocalałych według płci.
2. `surv_by_class` – odsetek ocalałych według klasy.
3. `pivot` – tabela przestawna: wiersze `sex`, kolumny `pclass`, wartości: odsetek ocalałych.
4. `pivot_n` – ta sama tabela, ale z **liczbą pasażerów** w każdej komórce (`aggfunc="count"`).

In [ ]:
# Twoje rozwiązanie
surv_by_sex = df_clean.groupby("sex")["survived"].mean()
surv_by_class = df_clean.groupby("pclass")["survived"].mean()
pivot = df_clean.pivot_table(index="sex", columns="pclass", values="survived", aggfunc="mean")
pivot_n = df_clean.pivot_table(index="sex", columns="pclass", values="survived", aggfunc="count")

print(pivot)
print(pivot_n)

In [ ]:
# TEST
assert abs(surv_by_sex["female"] - 0.742) < 1e-3 and abs(surv_by_sex["male"] - 0.189) < 1e-3
assert abs(surv_by_class[1] - 0.630) < 1e-3 and abs(surv_by_class[3] - 0.242) < 1e-3
assert abs(pivot.loc["female", 1] - 0.968) < 1e-3 and abs(pivot.loc["male", 3] - 0.135) < 1e-3
assert pivot_n.loc["female", 3] == 144 and pivot_n.values.sum() == 891
print("C1 OK")

*Twoja odpowiedź:* która cecha lepiej rozdziela ocalałych od ofiar: płeć czy klasa? Która grupa z tabeli przestawnej odstaje od reguły „kobiety przeżyły”? Dlaczego warto patrzeć też na `pivot_n`?

Lepiej rozdziela plec, poniewaz latwo zauwazyc ze mniej przezylo mezczyzn z klasy 1 od kobiet z klasy 3. I ta klasa kobiet rowniez odstaje od reguly "kobiety przezyly". Warto patrzeć na pivot_n, bo pokazuje liczebność każdej grupy i pozwala ocenić, czy odsetki opierają się na wystarczającej liczbie obserwacji.

---
## Część D. Wykresy: `plt.subplots` i seaborn

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))   # 1 wiersz, 2 kolumny wykresów, rozmiar w calach
```
- `fig` to cały rysunek (zapisujemy go `fig.savefig(...)`),
- `axes` to tablica układów współrzędnych: `axes[0]` po lewej, `axes[1]` po prawej.

**seaborn** to nakładka na matplotlib dla danych w `DataFrame`: podajesz tabelę (`data=`), nazwy kolumn (`x=`, `y=`, `hue=` – kolor wg kategorii) i **na którym wykresie rysować** (`ax=`).

| Funkcja | Co rysuje |
|---|---|
| `sns.histplot(data=df, x="age", hue="survived", ax=...)` | histogram, osobno dla każdej wartości `hue` |
| `sns.countplot(data=df, x="pclass", hue="survived", ax=...)` | liczba wierszy w każdej kategorii |
| `sns.barplot(data=df, x="pclass", y="survived", hue="sex", ax=...)` | **średnia** `y` w każdej kategorii + pionowa kreska: przedział ufności |

### Zadanie D1. Jeden rysunek, dwa wykresy
1. Utwórz `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`.
2. Po lewej (`axes[0]`): `sns.histplot` wieku z `df_clean`, 20 przedziałów, kolor według `survived`.
3. Po prawej (`axes[1]`): `sns.barplot` odsetka ocalałych według klasy, z podziałem na płeć (`hue="sex"`). Dodaj poziomą przerywaną linię na wysokości ogólnego odsetka ocalałych (`axes[1].axhline(...)`).
4. Każdemu wykresowi nadaj tytuł i podpisz osie (`set_title`, `set_xlabel`, `set_ylabel`). Zapisz rysunek do `titanic.png` (`fig.savefig("titanic.png", dpi=150)`).

In [ ]:
# Twoje rozwiązanie
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(data=df_clean, x="age", hue="survived", bins=20, ax=axes[0])
axes[0].set_title("Rozkład wieku wg przeżycia")
axes[0].set_xlabel("Wiek")
axes[0].set_ylabel("Liczba pasażerów")

sns.barplot(data=df_clean, x="pclass", y="survived", hue="sex", ax=axes[1])
axes[1].axhline(df_clean["survived"].mean(), color="black", linestyle="--")
axes[1].set_title("Odsetek ocalałych wg klasy i płci")
axes[1].set_xlabel("Klasa")
axes[1].set_ylabel("Odsetek ocalałych")

fig.savefig("titanic.png", dpi=150)

In [ ]:
# TEST
import os
assert len(fig.axes) >= 2, "Rysunek powinien mieć dwa wykresy"
assert os.path.exists("titanic.png") and os.path.getsize("titanic.png") > 5000
assert all(ax.get_title() for ax in fig.axes[:2]), "Brakuje tytułów wykresów"
print("D1 OK")

*Twoja odpowiedź:* na histogramie słupki w okolicach 20–30 lat są wyraźnie wyższe od sąsiednich. Skąd się wzięły? Jak wyglądałby histogram, gdyby braki uzupełnić jedną medianą?

Wysoki słupek w okolicach 25-30 lat powstał przez uzupełnienie braków medianą w grupach (płeć, klasa). 177 uzupełnionych wartości skupiło się wokół kilku mediów grupowych, które leżą blisko siebie, więc wpadły do tych samych przedziałów histogramu i sztucznie je podbiły. Gdyby braki uzupełnić jedną medianą (ok. 28 lat), wszystkie 177 wartości trafiłoby do jednego przedziału i słupek byłby jeszcze wyższy i węższy, a rozkład wieku jeszcze bardziej zniekształcony.

---
## Część E. Pierwszy klasyfikator

Porównamy trzy klasyfikatory:

1. **Baseline „nikt nie przeżył”** – zawsze 0. To punkt odniesienia: każdy sensowny model musi być od niego lepszy.
2. **Reguła „kobieta → 1”** – wymyślona przez człowieka.
3. **Klasyfikator z tabeli przestawnej** – reguła **wyznaczona z danych**: dla każdej grupy (płeć, klasa) odczytujemy z `pivot` odsetek ocalałych i przewidujemy 1, jeśli przekracza 0.5. To najprostsza forma uczenia: model „zapamiętuje”, co zwykle działo się w danej grupie.

Do przypisania każdemu pasażerowi wartości jego grupy przyda się `transform` (jak w B2):
```python
rate = df_clean.groupby(["sex", "pclass"])["survived"].transform("mean")   # wartość z pivot dla każdego wiersza
```

### Zadanie E1. Trzy klasyfikatory
1. `y` – kolumna `survived` jako tablica NumPy (`to_numpy()`).
2. `y_zero`, `y_rule`, `y_pivot` – predykcje trzech klasyfikatorów (tablice liczb całkowitych 0/1).
3. `m_zero`, `m_rule`, `m_pivot` – ich metryki (`binary_metrics`).
4. `df_models` – tabela z wierszami `"zawsze-zero"`, `"kobieta → 1"`, `"tabela przestawna"` i kolumnami `TP`, `FP`, `FN`, `TN`, `accuracy`, `precision`, `recall`, `f1`.
5. Sprawdź jeden z wyników funkcją `classification_report` ze `sklearn.metrics`.

In [ ]:
from sklearn.metrics import classification_report

# Twoje rozwiązanie

import numpy as np
import pandas as pd

y = df_clean["survived"].to_numpy()

y_zero = np.zeros(len(df_clean), dtype=int)

y_rule = (df_clean["sex"] == "female").astype(int).to_numpy()

rate = df_clean.groupby(["sex", "pclass"])["survived"].transform("mean")
y_pivot = (rate > 0.5).astype(int).to_numpy()

m_zero = binary_metrics(y, y_zero)
m_rule = binary_metrics(y, y_rule)
m_pivot = binary_metrics(y, y_pivot)

cols = ["TP", "FP", "FN", "TN", "accuracy", "precision", "recall", "f1"]
df_models = pd.DataFrame(
    [m_zero, m_rule, m_pivot],
    index=["zawsze-zero", "kobieta → 1", "tabela przestawna"],
)[cols]

print(df_models)

print(classification_report(y, y_pivot))

In [ ]:
# TEST
assert m_zero["recall"] == 0.0 and abs(m_zero["accuracy"] - 0.6162) < 1e-3
assert (m_rule["TP"], m_rule["FP"], m_rule["FN"], m_rule["TN"]) == (233, 81, 109, 468)
assert (m_pivot["TP"], m_pivot["FP"], m_pivot["FN"], m_pivot["TN"]) == (161, 9, 181, 540)
assert list(df_models.index) == ["zawsze-zero", "kobieta → 1", "tabela przestawna"]
assert {"accuracy", "precision", "recall", "f1"} <= set(df_models.columns)
print("E1 OK")

### Zadanie E2. Wnioski
Odpowiedz krótko, odwołując się do liczb z `df_models`:

1. O ile oba klasyfikatory poprawiają accuracy względem baseline'u? Dlaczego samo „accuracy = 0.79” bez baseline'u niewiele mówi?
2. Reguła „kobieta → 1” i klasyfikator z tabeli przestawnej mają **identyczną** accuracy. Czym się różnią? Kim są osoby, które klasyfikator z tabeli przestawnej „zmienił” w porównaniu z regułą?
3. Który z nich wybrał(a)byś, gdyby fałszywy alarm (FP) był bardzo kosztowny? A gdyby kosztowne było przeoczenie (FN)?
4. Klasyfikator z tabeli przestawnej „nauczył się” na tych samych danych, na których go oceniamy. Dlaczego taka ocena jest zbyt optymistyczna? Co by się stało, gdyby grup było bardzo dużo, a w każdej po kilka osób?
5. Te reguły decydują na podstawie płci. Podaj zastosowanie, w którym taka reguła byłaby niedopuszczalna niezależnie od accuracy, i wyjaśnij dlaczego.

*Twoja odpowiedź:*
1. Baseline ma accuracy 0,616, a oba klasyfikatory 0,787, więc poprawa to ok. 0,17. Samo „0,79” mało mówi, bo nie wiadomo, czy to dużo, dopóki nie porównamy z modelem, który nic nie robi. Tu 61,6% pasażerów zginęło, więc wystarczy zawsze zgadywać 0 i już mamy 62%, mimo że recall wynosi 0 (żadnego ocalałego nie wykrywamy). Accuracy zależy od tego, jak liczne są klasy.

2. Różnią się tym, co poświęcają. Reguła „kobieta = 1” ma precision 0,742 i recall 0,681 (TP 233, FP 81, FN 109). Tabela przestawna ma precision 0,947 i recall 0,471 (TP 161, FP 9, FN 181). Zmienione zostały kobiety z 3. klasy: ich odsetek przeżycia to dokładnie 0,5, a to nie jest więcej niż 0,5, więc model dał im 0 zamiast 1. Dzięki temu FP spadło z 81 do 9, ale FN wzrosło ze 109 do 181.

3. Jak drogi jest fałszywy alarm (FP), to biorę tabelę przestawną, bo ma tylko 9 FP i precision 0,95. Jak drogie jest przeoczenie (FN), to biorę regułę „kobieta = 1”, bo ma mniej FN (109 zamiast 181) i wyższy recall (0,68 vs 0,47).

4. Model uczył się na tych samych danych, na których go oceniamy, więc w pewnym sensie zna już odpowiedzi i wynik wychodzi zawyżony. Nie wiadomo, jak by się spisał na nowych pasażerach. Gdyby grup było bardzo dużo, a w każdej tylko kilka osób, to odsetek w grupie byłby dość przypadkowy (np. 1 z 3 przeżyła, czyli 0,33). Model by po prostu zapamiętał pojedyncze przypadki, czyli by się przeuczył. Na danych treningowych wyglądałby świetnie, a na nowych znacznie gorzej. Dlatego powinno się dzielić dane na treningowe i testowe.

5. Na przykład przy przyznawaniu kredytów albo w rekrutacji. Decydowanie o kimś na podstawie płci to dyskryminacja, która jest zakazana prawnie i nieetyczna, nawet jeśli reguła podnosi accuracy. To, że w danych jest jakaś zależność, nie znaczy, że wolno oceniać człowieka według jego grupy.

---
## Część F. Zadania dodatkowe (dla chętnych / do domu)

**F1. Kobiety i dzieci.** Zbuduj regułę „kobieta **lub** dziecko poniżej 10 lat → 1”, policz metryki i dopisz wiersz do `df_models`. Co się poprawiło, a co pogorszyło? Ile dzieci z brakującym wiekiem reguła „przeoczyła”, bo dostały medianę grupy (wskazówka: `age_missing`)?

**F2. Więcej grup = lepiej?** Dodaj kolumnę `age_group` (`pd.cut(df_clean["age"], bins=[0, 10, 18, 40, 60, 100])`) i zbuduj klasyfikator z tabeli przestawnej dla grup (płeć, klasa, `age_group`). Policz accuracy. Ile grup ma mniej niż 10 osób? Wróć do pytania E2.4.

**F3. Próg na cenę biletu.** Potraktuj cenę biletu jako „wynik modelu”: `y_hat = (df_clean["fare"] >= t)`. Dla 50 progów `t` od 0 do 100 policz precision, recall i F1, narysuj krzywe w funkcji `t` (seaborn albo matplotlib). Przy jakim progu F1 jest największe?

In [ ]:
# Twoje rozwiązanie